# Notebook 04 — Agrupar y agregar (`groupby`)

Hasta ahora aprendiste a **inspeccionar**, **filtrar** y **limpiar** datos. Hoy das un salto: vas a **resumir** datos por grupos. 📊

`groupby` es una de las herramientas más poderosas de pandas y se usa todos los días en análisis de datos. La idea se llama **split-apply-combine**:

1. **Split** — divides el DataFrame en grupos según una columna (ej. por especie).
2. **Apply** — aplicas una función a cada grupo (ej. promedio, suma, conteo).
3. **Combine** — pandas junta los resultados en una nueva tabla.

## Objetivos de aprendizaje

1. Entender el patrón **split-apply-combine**.
2. Agrupar por una columna y aplicar agregaciones (`.mean()`, `.sum()`, `.count()`, `.median()`).
3. Agrupar por **múltiples columnas** a la vez.
4. Aplicar **varias agregaciones** simultáneamente con `.agg()`.
5. Usar `.value_counts()` como atajo para conteos por categoría.

---

## 1. Repaso rápido del Notebook 03

| Operación | Sintaxis |
|---|---|
| Detectar NaN | `df.isna().sum()` |
| Eliminar filas con NaN | `df.dropna()` |
| Rellenar con constante | `s.fillna("Unknown")` |
| Rellenar con mediana | `s.fillna(s.median())` |

---

## 2. Setup

Cargamos `penguins`. Para evitar ruido por valores faltantes, partimos del dataset **sin NaN** (ya sabes hacerlo desde el Notebook 03).

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns

df = sns.load_dataset("penguins").dropna().reset_index(drop=True)
print(f"Dataset: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

---

## 3. Split-apply-combine intuitivo

Imagina que tienes una caja de fichas de colores y quieres saber **cuántas fichas hay de cada color**.

1. **Split** — separas las fichas en pilas por color (rojas, azules, verdes).
2. **Apply** — cuentas cuántas hay en cada pila.
3. **Combine** — anotas el resultado en una tabla final.

`groupby` hace exactamente eso, pero con filas de un DataFrame.

### Sintaxis básica

```python
df.groupby("col_para_agrupar")["col_a_agregar"].mean()
```

- `groupby("col_para_agrupar")` → divide el DataFrame en grupos.
- `["col_a_agregar"]` → la columna sobre la que aplicamos la operación.
- `.mean()` → la función de agregación (también `.sum()`, `.count()`, `.median()`, `.min()`, `.max()`...).

### Demo

In [ ]:
# Average bill length per species
df.groupby("species")["bill_length_mm"].mean()

In [ ]:
# Count of penguins per species (using .size())
df.groupby("species").size()

### 🏋️ Ejercicio 1 — `groupby` + `.mean()`

Crea una `Series` llamada **`mean_body_mass_by_species`** que contenga el **peso corporal promedio** (`body_mass_g`) **por especie**.

💡 Tip: `df.groupby("species")["body_mass_g"].mean()`.

In [ ]:
# YOUR CODE HERE
mean_body_mass_by_species = None


In [ ]:
# Tests
assert isinstance(mean_body_mass_by_species, pd.Series), "mean_body_mass_by_species must be a pandas Series"
assert mean_body_mass_by_species.index.name == "species", "Index should be named 'species'"
assert len(mean_body_mass_by_species) == 3, f"Expected 3 species, got {len(mean_body_mass_by_species)}"
assert set(mean_body_mass_by_species.index) == {"Adelie", "Chinstrap", "Gentoo"}, "Species names mismatch"
assert np.isclose(mean_body_mass_by_species["Adelie"], 3706.16, atol=1.0), \
    f"Adelie mean should be ~3706.16, got {mean_body_mass_by_species['Adelie']:.2f}"
assert np.isclose(mean_body_mass_by_species["Gentoo"], 5092.43, atol=1.0), \
    f"Gentoo mean should be ~5092.43, got {mean_body_mass_by_species['Gentoo']:.2f}"

print("✅ ¡Bien! Calculaste el peso promedio por especie.")
print(mean_body_mass_by_species)

---

## 4. Varias agregaciones a la vez con `.agg()`

A veces quieres más de un estadístico por grupo (mín, máx, media...). Usamos `.agg()` pasando una lista de funciones:

```python
df.groupby("col")["valor"].agg(["min", "max", "mean"])
```

El resultado es un `DataFrame` (no una `Series`) donde cada columna es una agregación.

### Demo

In [ ]:
# Min, max, and mean of bill_length_mm per species
df.groupby("species")["bill_length_mm"].agg(["min", "max", "mean"])

### 🏋️ Ejercicio 2 — `.agg()` con varias funciones

Crea un `DataFrame` llamado **`flipper_stats_by_species`** con las columnas `min`, `max` y `mean` del **largo de aleta** (`flipper_length_mm`) **por especie**.

In [ ]:
# YOUR CODE HERE
flipper_stats_by_species = None


In [ ]:
# Tests
assert isinstance(flipper_stats_by_species, pd.DataFrame), "flipper_stats_by_species must be a DataFrame"
assert flipper_stats_by_species.shape == (3, 3), f"Expected shape (3, 3), got {flipper_stats_by_species.shape}"
assert list(flipper_stats_by_species.columns) == ["min", "max", "mean"], \
    f"Columns must be ['min', 'max', 'mean'], got {list(flipper_stats_by_species.columns)}"
assert flipper_stats_by_species.loc["Adelie", "min"] == 172.0, "Adelie min flipper should be 172.0"
assert flipper_stats_by_species.loc["Gentoo", "max"] == 231.0, "Gentoo max flipper should be 231.0"
assert np.isclose(flipper_stats_by_species.loc["Chinstrap", "mean"], 195.82, atol=0.5), \
    f"Chinstrap mean flipper should be ~195.82, got {flipper_stats_by_species.loc['Chinstrap', 'mean']:.2f}"

print("✅ ¡Excelente! Aplicaste tres agregaciones a la vez.")
flipper_stats_by_species

---

## 5. Agrupar por **varias** columnas

Puedes pasar una **lista** a `groupby` para agrupar por más de una columna. El resultado tiene un **MultiIndex** (índice jerárquico):

```python
df.groupby(["col1", "col2"])["valor"].mean()
```

Esto responde preguntas más finas, por ejemplo:
> *¿Cuál es el peso promedio por **especie y sexo**?*

### Demo

In [ ]:
# Mean bill length grouped by species and sex
df.groupby(["species", "sex"])["bill_length_mm"].mean()

### 🏋️ Ejercicio 3 — groupby multi-columna

Crea una `Series` llamada **`mass_by_species_sex`** con el **peso corporal promedio** (`body_mass_g`) **por `species` y `sex`** (en ese orden).

In [ ]:
# YOUR CODE HERE
mass_by_species_sex = None


In [ ]:
# Tests
assert isinstance(mass_by_species_sex, pd.Series), "mass_by_species_sex must be a pandas Series"
assert isinstance(mass_by_species_sex.index, pd.MultiIndex), "Index must be a MultiIndex"
assert mass_by_species_sex.index.names == ["species", "sex"], \
    f"Index names must be ['species', 'sex'], got {mass_by_species_sex.index.names}"
assert len(mass_by_species_sex) == 6, f"Expected 6 (species, sex) groups, got {len(mass_by_species_sex)}"
assert np.isclose(mass_by_species_sex.loc[("Adelie", "Female")], 3368.84, atol=1.0), \
    f"Adelie/Female mean should be ~3368.84, got {mass_by_species_sex.loc[('Adelie', 'Female')]:.2f}"
assert np.isclose(mass_by_species_sex.loc[("Gentoo", "Male")], 5484.84, atol=1.0), \
    f"Gentoo/Male mean should be ~5484.84, got {mass_by_species_sex.loc[('Gentoo', 'Male')]:.2f}"

print("✅ ¡Genial! Agrupaste por dos columnas.")
print(mass_by_species_sex)

---

## 6. `.value_counts()` — conteo rápido por categoría

Si solo quieres **contar cuántas filas hay de cada valor** en una columna, hay un atajo más corto que `groupby(...).size()`: el método **`.value_counts()`**.

```python
df["col"].value_counts()                 # conteo descendente por defecto
df["col"].value_counts(ascending=True)   # ascendente
df["col"].value_counts(normalize=True)   # proporciones (suman 1.0)
```

### Demo

In [ ]:
# Count penguins by sex
df["sex"].value_counts()

### 🏋️ Ejercicio 4 — `value_counts`

Crea una `Series` llamada **`island_counts`** con el conteo de pingüinos **por isla**, ordenado de mayor a menor (que es el comportamiento por defecto).

In [ ]:
# YOUR CODE HERE
island_counts = None


In [ ]:
# Tests
assert isinstance(island_counts, pd.Series), "island_counts must be a pandas Series"
assert len(island_counts) == 3, f"Expected 3 islands, got {len(island_counts)}"
assert set(island_counts.index) == {"Biscoe", "Dream", "Torgersen"}, "Island names mismatch"
assert island_counts.index[0] == "Biscoe", f"Most frequent island should be 'Biscoe', got {island_counts.index[0]}"
assert island_counts.iloc[0] >= island_counts.iloc[-1], "Series should be sorted descending"
assert island_counts.sum() == len(df), "Counts should sum to total number of rows"

print("✅ ¡Bien! Contaste pingüinos por isla.")
print(island_counts)

---

## 7. Combinar **filtrado** + **groupby**

`groupby` se combina muy bien con todo lo aprendido en notebooks anteriores. Por ejemplo: **primero filtras**, **luego agrupas**.

> *De los pingüinos que viven en la isla **Biscoe**, ¿cuál es el peso promedio por especie?*

### Demo

In [ ]:
# Filter first, then group
biscoe_penguins = df[df["island"] == "Biscoe"]
biscoe_penguins.groupby("species")["bill_length_mm"].mean()

### 🏋️ Ejercicio 5 — filtrado + groupby

Crea una `Series` llamada **`biscoe_mass_by_species`** con el **peso corporal promedio** (`body_mass_g`) **por especie**, considerando **solo los pingüinos de la isla `"Biscoe"`**.

💡 Tip: filtra primero con una máscara booleana (Notebook 02), luego aplica `groupby`.

In [ ]:
# YOUR CODE HERE
biscoe_mass_by_species = None


In [ ]:
# Tests
assert isinstance(biscoe_mass_by_species, pd.Series), "biscoe_mass_by_species must be a pandas Series"
assert biscoe_mass_by_species.index.name == "species", "Index should be named 'species'"
assert set(biscoe_mass_by_species.dropna().index) >= {"Adelie", "Gentoo"}, \
    "Biscoe should contain at least Adelie and Gentoo"
assert "Chinstrap" not in biscoe_mass_by_species.dropna().index, \
    "Chinstrap doesn't live on Biscoe — your result shouldn't include it (or should be NaN)"
assert np.isclose(biscoe_mass_by_species["Gentoo"], 5092.43, atol=1.0), \
    f"Biscoe Gentoo mean should be ~5092.43, got {biscoe_mass_by_species['Gentoo']:.2f}"
assert np.isclose(biscoe_mass_by_species["Adelie"], 3709.66, atol=1.0), \
    f"Biscoe Adelie mean should be ~3709.66, got {biscoe_mass_by_species['Adelie']:.2f}"

print("✅ ¡Excelente! Combinaste filtrado y groupby.")
print(biscoe_mass_by_species)

---

## 8. Resumen — ¿qué aprendiste?

🎉 ¡Súper! Ya dominas el patrón **split-apply-combine**, una de las herramientas más usadas en análisis de datos.

| Operación | Sintaxis |
|---|---|
| Una agregación por grupo | `df.groupby("col")["valor"].mean()` |
| Varias agregaciones a la vez | `df.groupby("col")["valor"].agg(["min","max","mean"])` |
| Agrupar por varias columnas | `df.groupby(["col1","col2"])["valor"].mean()` |
| Conteo por categoría (atajo) | `df["col"].value_counts()` |
| Proporciones | `df["col"].value_counts(normalize=True)` |
| Filtrar + agrupar | `df[mask].groupby("col")["valor"].mean()` |

### Otras agregaciones útiles

`sum`, `count`, `min`, `max`, `median`, `std`, `var`, `first`, `last`, `nunique`.

## ¿Qué viene en el próximo notebook?

En **Notebook 05 — Combinar DataFrames (`merge`, `concat`)** vas a aprender a **unir tablas** que comparten una clave común. Es la base para trabajar con datos que vienen de varias fuentes (como cuando haces un VLOOKUP en Excel o un JOIN en SQL). ¡Nos vemos ahí! 🔗